# NEURO 120: Bellier extension supplement

This notebook is **Bellier et al. (2023)** follow-on work: vocal versus instrumental decoding on the continuous supergrid, simple nulls, and figures that are not the primary Norman–Haignere driver. The full paper pipeline lives in `neuro120_main_reproduction.ipynb`.

**Conventions**

- Add `logic/` to `sys.path` the same way as the main notebook so imports match `python logic/pipeline.py`.
- Random seeds use `config.RANDOM_STATE`, consistent with `logic/pipeline.py` and the main notebook.
- `_lap` and `print_run_timings()` record wall time between sections. Run the timing cell at the end after the blocks you care about.


## 1. Setup

Imports, add `logic/` to the path, load the Norman–Haignere dataset for reference, and create output folders via `config.ensure_dirs()`. This cell also defines `_lap` and `print_run_timings()` so each analysis section records wall time; run the **Timing summary** cell at the very end after a full pass.

In [12]:
# Uncomment if dependencies are missing in this kernel:
# %pip install -r requirements.txt

import importlib
import sys
import time
import warnings
import wave
from pathlib import Path

import numpy as np
from IPython.display import Image, display

# Notebooks are not inside the logic package, so add logic/ to sys.path
# for the same flat imports as python logic/pipeline.py (import config, etc.).
LOGIC_DIR = (Path.cwd() / "logic").resolve()
if str(LOGIC_DIR) not in sys.path:
    sys.path.insert(0, str(LOGIC_DIR))

import config
import pipeline
from bellier_data import build_supergrid, electrode_subsets, load_vocal_segments
from bellier_decoder import run_vocal_instrumental_decoder
from config import (
    BELLIER_AUDIO_PATH,
    BELLIER_FS,
    BELLIER_T,
    BELLIER_VOCAL_CSV,
    FIG_DIR,
)
from data_utils import build_dataset
from plots import (
    plot_bellier_matched_control_histogram,
    plot_bellier_top7_vs_random_histogram,
)

warnings.filterwarnings("ignore")

# Wall-clock timing for each notebook section (call _lap at end of each block).
_t_mark = time.perf_counter()
_RUN_SECTIONS = []


def _lap(label):
    global _t_mark
    now = time.perf_counter()
    _RUN_SECTIONS.append((label, now - _t_mark))
    _t_mark = now


def print_run_timings():
    if not _RUN_SECTIONS:
        print("No section timings recorded.")
        return
    total = sum(dt for _, dt in _RUN_SECTIONS)
    print("\nSection run times (wall clock)")
    for name, dt in _RUN_SECTIONS:
        print(f"  {name:44s}  {dt:8.2f} s")
    print("  " + "-" * 56)
    print(f"  {'Total':44s}  {total:8.2f} s")


config.ensure_dirs()
ds = build_dataset()
print("meta:", ds["meta"])
_lap("1. Setup and dataset load")


meta: {'n_electrodes': 33, 'n_stimuli': 49, 'n_time': 201, 'dt_s': 0.010000000000000009, 'time_window_s': (0.0, 2.0), 'n_per_group': {'song': 7, 'speech': 15, 'music': 11}, 'n_per_class': {'song': 11, 'speech': 17, 'music': 21}, 'keep_classes': ['song', 'speech', 'music']}


## 2. Stimulus alignment and vocal mask

**Question.** Does the WAV length match the neural series at 100 Hz, and does the hand-built vocal mask look reasonable?

**Method.** Read `thewall1.wav`, compare duration to `BELLIER_T` and `BELLIER_FS`, then parse `vocal_segments.csv` via `load_vocal_segments()` and print merged intervals.

In [13]:
# Check WAV length versus neural T at 100 Hz, then print vocal mask coverage.
with wave.open(str(BELLIER_AUDIO_PATH), "rb") as w:
    wav_dur = w.getnframes() / w.getframerate()
neural_dur = BELLIER_T / BELLIER_FS
print(f"WAV duration: {wav_dur:.3f} s ({BELLIER_AUDIO_PATH.name})")
print(f"Neural duration: {neural_dur:.3f} s (T={BELLIER_T} at {BELLIER_FS} Hz)")
print(f"Aligned within 10 ms: {np.isclose(wav_dur, neural_dur, atol=0.01)}")

mask = load_vocal_segments()
diff = np.diff(mask.astype(int))
starts = np.where(diff == 1)[0] + 1
ends = np.where(diff == -1)[0] + 1
if mask[0]:
    starts = np.r_[0, starts]
if mask[-1]:
    ends = np.r_[ends, len(mask)]
print(f"\nVocal CSV: {BELLIER_VOCAL_CSV.name}")
print(f"Vocal samples: {int(mask.sum())} / {len(mask)} ({mask.mean()*100:.2f}%)")
print(f"Merged intervals: {len(starts)}")
for s, e in zip(starts, ends):
    print(
        f"  {s/BELLIER_FS:7.3f} to {e/BELLIER_FS:7.3f} s "
        f"(duration {(e-s)/BELLIER_FS:.3f} s)"
    )

_lap("2. Stimulus alignment and vocal mask")


WAV duration: 190.720 s (thewall1.wav)
Neural duration: 190.720 s (T=19072 at 100 Hz)
Aligned within 10 ms: True

Vocal CSV: vocal_segments.csv
Vocal samples: 5803 / 19072 (30.43%)
Merged intervals: 4
   15.080 to  16.900 s (duration 1.820 s)
   17.460 to  19.150 s (duration 1.690 s)
   24.410 to  29.570 s (duration 5.160 s)
   42.660 to  92.020 s (duration 49.360 s)


## 3. Bellier supergrid

**Method.** `build_supergrid()` loads or builds the pooled HFA tensor used by Bellier helpers in `pipeline`. The next cell calls `importlib.reload(pipeline)` so edits to `logic/pipeline.py` apply without restarting the kernel; re-run that cell after you change the file.

In [14]:
importlib.reload(pipeline)

supergrid = build_supergrid()
_lap("3. Build supergrid")


## 4. Focal electrode indices

**Question.** Which supergrid columns define the focal set for null demos?

**Method.** Take `right_STG` from `electrode_subsets`, then the first `min(7, n)` indices so the focal set size matches the Norman song-selective count when possible. This matches the anatomical grouping used in the main Bellier decoder section.

In [15]:
subs = electrode_subsets(supergrid)
rh = np.asarray(subs["right_STG"], dtype=int)
subset_size = int(min(7, rh.size))
true_subset = rh[:subset_size]
print("true_subset (right_STG):", true_subset, "len =", subset_size)
_lap("4. Right STG focal indices")


true_subset (right_STG): [ 0  1  5  6  9 10 14] len = 7


## 5. Top seven versus random null (balanced accuracy)

**Question.** How does focal-set balanced accuracy compare to random same-size draws from the full electrode pool?

**Method.** `pipeline.run_bellier_top7_vs_random_avg` with `n_random` Monte Carlo draws. The cell below prints the summary table, builds the histogram, and shows the PNG under `results/figures/`.

In [ ]:
bellier_compare = pipeline.run_bellier_top7_vs_random_avg(
    true_subset,
    n_random=100,
    seed=config.RANDOM_STATE,
)
display(bellier_compare["summary"])

rand = bellier_compare["random_bacc"]
top = bellier_compare["top_bacc"]
plot_bellier_top7_vs_random_histogram(
    np.asarray(rand),
    float(top),
    stem="fig_part2_bellier_top7_vs_random",
)
display(
    Image(
        filename=str(FIG_DIR / "fig_part2_bellier_top7_vs_random.png"),
        width=480,
    )
)
_lap("5. Top7 vs random null")


  [decoder] subset=true_subset n_elec=    7 n_wins= 1903 pos_rate=0.304
    logreg  bacc=0.624 CI=[0.591, 0.630]
    cnn     bacc=0.602 CI=[0.563, 0.610] (n_elec_in=7)
  [decoder] subset=rand_subset n_elec=    7 n_wins= 1903 pos_rate=0.304
    logreg  bacc=0.556 CI=[0.538, 0.571]
    cnn     bacc=0.581 CI=[0.549, 0.593] (n_elec_in=7)
  [decoder] subset=rand_subset n_elec=    7 n_wins= 1903 pos_rate=0.304
    logreg  bacc=0.545 CI=[0.518, 0.543]
    cnn     skipped (logreg bacc 0.545 < chance+0.05 = 0.550)
  [decoder] subset=rand_subset n_elec=    7 n_wins= 1903 pos_rate=0.304
    logreg  bacc=0.577 CI=[0.547, 0.586]
    cnn     bacc=0.594 CI=[0.563, 0.608] (n_elec_in=7)
  [decoder] subset=rand_subset n_elec=    7 n_wins= 1903 pos_rate=0.304
    logreg  bacc=0.489 CI=[0.483, 0.499]
    cnn     skipped (logreg bacc 0.489 < chance+0.05 = 0.550)
  [decoder] subset=rand_subset n_elec=    7 n_wins= 1903 pos_rate=0.304
    logreg  bacc=0.577 CI=[0.544, 0.577]
    cnn     bacc=0.580 CI=[0.563,

## 6. Vocal versus instrumental decoder on focal subset

**Method.** `run_vocal_instrumental_decoder` on `{"top7": true_subset}` with the vocal mask from disk. Requires `supergrid` and `true_subset` from earlier cells.

**CNN note.** Logistic regression always runs. The TinyTemporalCNN runs only if mean balanced accuracy from log-reg is at least `0.5 + config.CNN_MIN_BACC_OVER_CHANCE` (default 0.55); otherwise the console prints `cnn skipped` and `summary` has no `cnn` row for that subset. Same rule and rationale as in `writeup.tex` (Bellier methods, blocked-time CV paragraph).

In [ ]:
vocal_mask = load_vocal_segments()

top_res = run_vocal_instrumental_decoder(
    supergrid,
    {"top7": true_subset},
    vocal_mask,
    subsets=["top7"],
    seed=config.RANDOM_STATE,
)

print(top_res["summary"].columns.tolist())
display(top_res["summary"])
_lap("6. Vocal decoder on focal subset")


## 7. Matched random subset control (illustrative)

**Question.** Where does the focal subset sit relative to a small null of random same-size subsets?

**Method.** This block can be run alone: it rebuilds `supergrid`, recomputes `true_subset`, runs `run_bellier_matched_random_subset_control` with a **small** `n_subsets` for speed, overlays the main-style histogram, and prints an empirical p-value. Raise `n_subsets` toward hundreds for a stable null when you have time.

In [ ]:
supergrid = build_supergrid(cache=True, verbose=False)
vocal_mask = load_vocal_segments()
subs = electrode_subsets(supergrid)
rh = np.asarray(subs["right_STG"], dtype=int)
subset_size = int(min(7, rh.size))
true_subset = rh[:subset_size]

bellier_null = pipeline.run_bellier_matched_random_subset_control(
    true_subset,
    n_subsets=5,
    subset_size=subset_size,
    seed=config.RANDOM_STATE,
)
display(bellier_null["summary"])

null_scores = bellier_null["null_scores"]
true_score = bellier_null["true_score"]
null_mean = float(np.mean(null_scores))

all_res = run_vocal_instrumental_decoder(
    supergrid,
    {"all": np.arange(supergrid["hfa"].shape[1])},
    vocal_mask,
    subsets=["all"],
    seed=config.RANDOM_STATE,
)
all_bacc = float(
    all_res["summary"].query("model == 'logreg'")["mean_bacc"].iloc[0]
)

plot_bellier_matched_control_histogram(
    null_scores,
    true_score,
    all_electrodes_bacc=all_bacc,
    null_mean=null_mean,
    stem="fig_part2_bellier_matched_control",
)
display(
    Image(
        filename=str(FIG_DIR / "fig_part2_bellier_matched_control.png"),
        width=480,
    )
)

p = (1 + np.sum(null_scores >= true_score)) / (len(null_scores) + 1)
print("empirical p:", p)
_lap("7. Matched null and histogram")


## 8. Timing summary

Wall seconds for each section that called `_lap` since the previous `_lap`. The **Total** line sums those intervals for this run. Execute after the cells above.

In [ ]:
print_run_timings()